# Week 2 — Multilingual Sentiment, MBERT (bert-base-multilingual-cased)
**Project**: Quantum Kernels — Multilingual + KTA-predicts-gap analysis

**The novel framing:** instead of "does it work in language X", this tests whether
the quantum-vs-classical gap VARIES across languages and whether **kernel-target
alignment (KTA) predicts that variation**. Each language gives one (KTA, gap) point;
correlating across 6 typologically diverse languages is the contribution.

**Languages (6, diverse scripts/families):**
English, Spanish, German (Latin script) + Arabic, Chinese, Hindi (other scripts)

**This notebook (Week 2):** for each language, extract MBERT embeddings,
PCA->8, classical baselines, AND the classical-kernel KTA. The quantum kernels
(and quantum KTA) come in the per-language Week 3 runs.

**Encoder:** bert-base-multilingual-cased (768-dim, first-token pooling)
**Metric:** macro F1 (3-class sentiment) | **Runtime:** ~10 min for all 6 languages

In [ ]:
!pip install datasets transformers torch scikit-learn numpy --quiet
import numpy as np, time, os, pickle, shutil, json
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

---
## Step 0: Storage + KTA helper

In [ ]:
TAG = 'multiling_mbert'
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")
def _persist_and_download(path):
    if os.path.abspath(SAVE_DIR) != os.getcwd():
        shutil.copy(path, os.path.join(SAVE_DIR, os.path.basename(path)))
        print(f"  saved -> {SAVE_DIR}/{os.path.basename(path)}")
    try:
        from google.colab import files
        files.download(path)
        print(f"  download -> {os.path.basename(path)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — saved above)")
def checkpoint(filename, save_fn):
    save_fn(filename); _persist_and_download(filename)
def save_figure(fig, name):
    for ext in ('png','pdf'):
        p = f"{name}.{ext}"; fig.savefig(p, bbox_inches='tight', dpi=150); _persist_and_download(p)
def _find(fname):
    dp = os.path.join(SAVE_DIR, fname); return dp if os.path.exists(dp) else fname
print("Helpers ready (Drive + download).")

def centered_kernel_alignment(K, y):
    """Kernel-target alignment between kernel matrix K and labels y.
    For multiclass, uses one-vs-rest ideal kernel averaged, via the
    standard centered alignment A(K, yy^T)."""
    y = np.asarray(y)
    classes = np.unique(y)
    # ideal kernel: 1 if same class, else 0 (handles multiclass)
    Y = np.zeros((len(y), len(y)))
    for c in classes:
        mask = (y == c).astype(float)
        Y += np.outer(mask, mask)
    # center both kernels
    n = K.shape[0]
    H = np.eye(n) - np.ones((n, n)) / n
    Kc = H @ K @ H
    Yc = H @ Y @ H
    num = np.sum(Kc * Yc)
    den = np.sqrt(np.sum(Kc * Kc) * np.sum(Yc * Yc))
    return float(num / den) if den > 0 else 0.0
print('KTA helper ready.')

---
## Step 1: Config — languages and encoder

In [ ]:
MODEL_NAME = 'bert-base-multilingual-cased'
LANGUAGES = ['english', 'spanish', 'german', 'arabic', 'chinese', 'hindi']
N_QUBITS = 8

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE); model.eval()
print(f"{MODEL_NAME} loaded (768-dim).")

def extract(texts, batch_size=32):
    out=[]
    for i in range(0,len(texts),batch_size):
        inp = tokenizer(texts[i:i+batch_size], padding=True, truncation=True,
                        max_length=128, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            o = model(**inp)
        out.append(o.last_hidden_state[:,0,:].cpu().numpy())   # first token
    return np.vstack(out)

---
## Step 2: Per-language extraction + baselines + classical KTA

Loops over all 6 languages. For each: load, extract embeddings, PCA->8,
classical baselines, and KTA of the best classical kernel (the RBF Gram matrix).
Everything saved per-language so Week 3 quantum runs can pick up each one.

In [ ]:
from datasets import load_dataset

per_lang = {}   # collects summary for the cross-language analysis later

for lang in LANGUAGES:
    print(f"\n{'='*50}\nLanguage: {lang}\n{'='*50}")
    ds = load_dataset("tyqiangz/multilingual-sentiments", lang)

    # collect balanced-ish train/test (3-class)
    def collect(split, n):
        texts=[s['text'] for s in ds[split].select(range(min(n, len(ds[split]))))]
        labels=np.array([s['label'] for s in ds[split].select(range(min(n, len(ds[split]))))])
        return texts, labels
    tr_texts, tr_labels = collect('train', 1300)
    te_split = 'test' if 'test' in ds else 'validation'
    te_texts, te_labels = collect(te_split, 300)

    print(f"  extracting {len(tr_texts)} train, {len(te_texts)} test...")
    tr_emb = extract(tr_texts); te_emb = extract(te_texts)

    # PCA -> 8 + scale
    pca = PCA(n_components=N_QUBITS, random_state=42).fit(tr_emb)
    scaler = MinMaxScaler(feature_range=(0,np.pi)); scaler.fit(pca.transform(tr_emb))
    Xtr_s = np.clip(scaler.transform(pca.transform(tr_emb)),0,np.pi)
    Xte_s = np.clip(scaler.transform(pca.transform(te_emb)),0,np.pi)

    # subsample to 1000/200
    n_tr=min(1000,len(Xtr_s)); n_te=min(200,len(Xte_s))
    itr=np.random.RandomState(42).choice(len(Xtr_s),n_tr,replace=False)
    ite=np.random.RandomState(42).choice(len(Xte_s),n_te,replace=False)
    X_tr,y_tr=Xtr_s[itr],tr_labels[itr]; X_te,y_te=Xte_s[ite],te_labels[ite]

    # classical baselines
    def mf1(yt,yp): return float(f1_score(yt,yp,average='macro'))
    baselines={}
    for name,clf in [('Linear',SVC(kernel='linear',C=1.0,random_state=42)),
                     ('RBF',SVC(kernel='rbf',C=1.0,gamma='scale',random_state=42)),
                     ('Polynomial',SVC(kernel='poly',C=1.0,degree=3,coef0=1,random_state=42))]:
        clf.fit(X_tr,y_tr); yp=clf.predict(X_te)
        baselines[name]={'accuracy':float(accuracy_score(y_te,yp)),'f1':mf1(y_te,yp)}
    best_classical_f1=max(v['f1'] for v in baselines.values())

    # classical KTA — RBF Gram matrix on the train set
    from sklearn.metrics.pairwise import rbf_kernel
    K_rbf = rbf_kernel(X_tr, X_tr, gamma=1.0/X_tr.shape[1])
    kta_classical = centered_kernel_alignment(K_rbf, y_tr)

    print(f"  best classical macro-F1: {best_classical_f1:.4f}  | classical KTA (RBF): {kta_classical:.4f}")

    # save per-language embeddings for Week 3 quantum run
    ltag=f'{TAG}_{lang}'
    checkpoint(f'{ltag}_embeddings.npz', lambda p,a=X_tr,b=y_tr,c=X_te,d=y_te: np.savez(
        p, X_tr_subset=a, y_tr_subset=b, X_te_subset=c, y_te_subset=d))
    checkpoint(f'{ltag}_baseline_results.pkl', lambda p,r=baselines: pickle.dump(r,open(p,'wb')))

    per_lang[lang]={'baselines':baselines,'best_classical_f1':best_classical_f1,
                    'kta_classical':kta_classical}

print("\nAll 6 languages extracted.")

---
## Step 3: Save the per-language summary (for the cross-language analysis)

In [ ]:
summary = {'encoder':'mbert','model':MODEL_NAME,'languages':LANGUAGES,
           'n_qubits':N_QUBITS,'metric':'macro','per_language':per_lang}
checkpoint(f'{TAG}_week2_summary.json', lambda p: json.dump(summary,open(p,'w'),indent=2))

print(f"\n{'Language':<12} {'BestClassF1':>12} {'ClassicalKTA':>13}")
print("-"*38)
for lang in LANGUAGES:
    d=per_lang[lang]
    print(f"{lang:<12} {d['best_classical_f1']:>12.4f} {d['kta_classical']:>13.4f}")
print("\nNext: run Week 3 quantum kernel per language (loads each _embeddings.npz).")
print("Week 3 records the quantum macro-F1 and quantum KTA, completing each (KTA, gap) point.")

---
## Week 2 done — Multilingual MBERT

### Saved per language (6 languages):
- `multiling_mbert_{lang}_embeddings.npz`, `multiling_mbert_{lang}_baseline_results.pkl`
### And:
- `multiling_mbert_week2_summary.json` — classical F1 + classical KTA per language

### Next — Week 3 (per language):
The quantum kernel run for each language produces the quantum macro-F1 and the
**quantum KTA**. Then the cross-language analysis correlates quantum KTA against
the quantum-classical gap across all 6 languages — the novel result. Same
analysis on both mBERT and XLM-R.